In [2]:
import os
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn

ROOT = os.environ.get("AGL_ROOT", ".")


def P(*p):
    return os.path.join(ROOT, *p)


OUT_DIR = P("Code Outputs", "GNN Outputs")
os.makedirs(OUT_DIR, exist_ok=True)

# ---- configuration identical to 17_gnn.ipynb
START, END = "1995-06-01", "2025-12-01"
TEST_MONTHS = 60
HORIZONS = [1, 3, 6, 12]
WINDOW = 12
H_HID, G_HID = 16, 24
EPOCHS, PATIENCE, LR, WD, DROP = 400, 30, 5e-3, 1e-4, 0.2
SEEDS = [0, 1, 2]
N_NULL = int(os.environ.get("N_NULL", 30))
NULL_SEED = 20260808

# SMOKE MODE - set GNN_SMOKE=1 to exercise every code path in about a minute
# with a single seed and 40 epochs. The numbers are meaningless; it exists only
# to prove the script runs end to end before committing to the real run.
SMOKE = os.environ.get("GNN_SMOKE") == "1"
if SMOKE:
    SEEDS, EPOCHS, PATIENCE, N_NULL = [0], 40, 40, 2
    print("*** SMOKE MODE - results are meaningless, code path test only ***")
SUF = "_SMOKE" if SMOKE else ""


def rmse(p, a):
    p, a = np.asarray(p, float), np.asarray(a, float)
    return np.sqrt(np.nanmean((p - a) ** 2))


def mae(p, a):
    p, a = np.asarray(p, float), np.asarray(a, float)
    return np.nanmean(np.abs(p - a))


# data: identical to 17_gnn.ipynb 
lev = pd.read_excel(P("Code Outputs", "Gap Interpolation Outputs",
                      "Unified_Interpolated_Levels.xlsx"))
lev["Date"] = pd.to_datetime(lev["Date"])
L = (lev.pivot(index="Date", columns="Reservoir", values="Level_m")
        .sort_index().asfreq("MS").loc[START:END])
LAKES = list(L.columns)
N = len(LAKES)
Lv = L.values
dL = np.diff(Lv, axis=0)
dates = L.index[1:]
T = len(dL)
split = T - TEST_MONTHS

mu, sd = dL[:split].mean(0), dL[:split].std(0) + 1e-8
dstd = ((dL - mu) / sd).astype(np.float32)
month = dates.month.values
sinm, cosm = np.sin(2 * np.pi * month / 12), np.cos(2 * np.pi * month / 12)
feats = np.stack([dstd,
                  np.repeat(sinm[:, None], N, 1),
                  np.repeat(cosm[:, None], N, 1)], axis=-1).astype(np.float32)
F_IN = feats.shape[-1]


def make_windows(lo, hi):
    X, Y = [], []
    for i in range(max(WINDOW, lo), hi):
        X.append(feats[i - WINDOW:i])
        Y.append(dstd[i])
    return np.array(X), np.array(Y)


val_start = int(split * 0.85)
Xtr, Ytr = make_windows(WINDOW, val_start)
Xva, Yva = make_windows(val_start, split)

# graphs
CORR = (pd.read_csv(P("Code Outputs", "EDA Outputs",
                      "EDA_06_corr_residual.csv"), index_col=0)
        .reindex(index=LAKES, columns=LAKES).values)
CCM = (pd.read_csv(P("Code Outputs", "CCM Outputs",
                     "CCM_09_adjacency_top3_diff_train.csv"), index_col=0)
       .reindex(index=LAKES, columns=LAKES).values.astype(float))
np.fill_diagonal(CCM, 0.0)


def sym_norm(A):
    """D^-1/2 A D^-1/2 with self-loops - the existing convention. Degrees use
    |A| so a signed graph still gets a positive, finite normaliser."""
    A = np.asarray(A, float) + np.eye(N)
    d = np.abs(A).sum(1)
    Di = np.diag(1.0 / np.sqrt(d))
    return torch.tensor(Di @ A @ Di, dtype=torch.float32)


def row_norm(A):
    """D^-1 A with self-loops. Each node's aggregation is a weighted MEAN over
    itself and its in-neighbours. Valid for directed graphs, unlike sym_norm."""
    A = np.asarray(A, float) + np.eye(N)
    d = np.abs(A).sum(1, keepdims=True)
    return torch.tensor(A / d, dtype=torch.float32)

_abs_asbuilt = np.abs(CORR).copy()
_abs_asbuilt[_abs_asbuilt < 0.2] = 0.0       

_abs = _abs_asbuilt.copy()
np.fill_diagonal(_abs, 0.0)                   

_signed = CORR.copy()
_signed[np.abs(_signed) < 0.2] = 0.0
np.fill_diagonal(_signed, 0.0)
_iu = np.triu_indices(N, 1)
assert (CORR[_iu] > 0).all(), "a negative residual correlation exists"
print(f"  residual correlations: all {len(_iu[0])} off-diagonal entries are "
      f"positive ({CORR[_iu].min():.3f} to {CORR[_iu].max():.3f}), so "
      f"|corr| discards no sign information")

GRAPHS = {
    "corr_abs_asbuilt": sym_norm(_abs_asbuilt),

    "corr_abs_sym":   sym_norm(_abs),

    "corr_abs_row":   row_norm(_abs),

    "ccm_directed":   row_norm(CCM),

    "ccm_symmetric":  sym_norm(np.maximum(CCM, CCM.T)),

    "corr_signed":    sym_norm(_signed),

    "identity":       torch.eye(N, dtype=torch.float32),
    "complete":       row_norm(1.0 - np.eye(N)),
}


# model
class GCN(nn.Module):
    def __init__(s, fin, fout):
        super().__init__()
        s.lin = nn.Linear(fin, fout)

    def forward(s, x, An):
        x = s.lin(x)
        x = torch.einsum("ij,bjf->bif", An, x)
        return torch.relu(x)


class STGNN(nn.Module):
    def __init__(s):
        super().__init__()
        s.g1 = GCN(F_IN, H_HID)
        s.g2 = GCN(H_HID, H_HID)
        s.gru = nn.GRU(H_HID, G_HID, batch_first=True)
        s.drop = nn.Dropout(DROP)
        s.out = nn.Linear(G_HID, 1)

    def forward(s, x, An):
        B, Lw, n, f = x.shape
        h = x.reshape(B * Lw, n, f)
        h = s.g2(s.g1(h, An), An)
        h = h.reshape(B, Lw, n, -1).permute(0, 2, 1, 3).reshape(B * n, Lw, -1)
        _, hn = s.gru(h)
        o = s.out(s.drop(hn[-1]))
        return o.reshape(B, n)


def train_one(seed, An):
    torch.manual_seed(seed)
    np.random.seed(seed)
    m = STGNN()
    opt = torch.optim.Adam(m.parameters(), lr=LR, weight_decay=WD)
    lossf = nn.MSELoss()
    xtr = torch.tensor(Xtr, dtype=torch.float32)
    ytr = torch.tensor(Ytr, dtype=torch.float32)
    xva = torch.tensor(Xva, dtype=torch.float32)
    yva = torch.tensor(Yva, dtype=torch.float32)
    best, best_state, wait = np.inf, None, 0
    for ep in range(EPOCHS):
        m.train()
        opt.zero_grad()
        loss = lossf(m(xtr, An), ytr)
        loss.backward()
        opt.step()
        m.eval()
        with torch.no_grad():
            vl = lossf(m(xva, An), yva).item()
        if vl < best - 1e-5:
            best, wait = vl, 0
            best_state = {k: v.clone() for k, v in m.state_dict().items()}
        else:
            wait += 1
            if wait >= PATIENCE:
                break
    m.load_state_dict(best_state)
    m.eval()
    return m


def evaluate(An):
    models = [train_one(s, An) for s in SEEDS]

    def predict_diff(w):
        x = torch.tensor(w[None], dtype=torch.float32)
        with torch.no_grad():
            return np.mean([mm(x, An).numpy()[0] for mm in models], axis=0)

    preds = {}
    for o in range(split, T):
        win = feats[o - WINDOW:o].copy()
        base = Lv[o]
        cum = 0.0
        Hh = min(max(HORIZONS), T - o)
        for k in range(Hh):
            pstd = predict_diff(win)
            cum = cum + (pstd * sd + mu)
            h = k + 1
            if h in HORIZONS:
                for j, lk in enumerate(LAKES):
                    preds[(lk, o + k, h)] = base[j] + cum[j]
            nxt = o + k
            mth = dates[nxt].month if nxt < T else ((dates[-1].month % 12) + 1)
            newf = np.stack([pstd,
                             np.full(N, np.sin(2 * np.pi * mth / 12), np.float32),
                             np.full(N, np.cos(2 * np.pi * mth / 12), np.float32)],
                            axis=-1)
            win = np.concatenate([win[1:], newf[None]], axis=0)

    rows = []
    for j, lk in enumerate(LAKES):
        for h in HORIZONS:
            Pv, Av = [], []
            for o in range(split, T):
                key = (lk, o + h - 1, h)
                if key in preds and (o + h) < len(Lv):
                    Pv.append(preds[key])
                    Av.append(Lv[o + h][j])
            rows.append({"Lake": lk, "Horizon_m": h,
                         "RMSE_m": round(rmse(Pv, Av), 4),
                         "MAE_m": round(mae(Pv, Av), 4),
                         "n_scored": len(Pv)})
    return pd.DataFrame(rows)


# ---- SARIMA denominator, from FC6 (which took it from the shared baseline) -
FC6 = pd.read_csv(os.path.join(OUT_DIR, "FC6_gnn_metrics.csv"))
SAR = FC6[FC6.Model == "SARIMA"].set_index(["Lake", "Horizon_m"])["RMSE_m"]


def add_skill(df):
    df["skill_vs_SARIMA_%"] = df.apply(
        lambda r: round(100 * (SAR[(r.Lake, r.Horizon_m)] - r.RMSE_m)
                        / SAR[(r.Lake, r.Horizon_m)], 1), axis=1)
    return df


print("=" * 74)
print("GNN ADJACENCY SENSITIVITY")
print("=" * 74)
print(f"\n  graphs: {', '.join(GRAPHS)}")
print(f"  null trials: {N_NULL}")

rows, summary = [], []
for gname, An in GRAPHS.items():
    met = add_skill(evaluate(An))
    met.insert(0, "Graph", gname)
    rows.append(met)
    mean = met.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()
    summary.append({"Graph": gname, **{f"h{h}": round(mean[h], 1)
                                       for h in HORIZONS}})
    print(f"    {gname:15s} " + "  ".join(
        f"h{h}={mean[h]:6.1f}" for h in HORIZONS))

SENS = pd.concat(rows, ignore_index=True)

# REGRESSION TEST
base = SENS[SENS.Graph == "corr_abs_asbuilt"][["Lake", "Horizon_m", "RMSE_m"]]
ref = FC6[FC6.Model == "GNN"][["Lake", "Horizon_m", "RMSE_m"]]
j = ref.merge(base, on=["Lake", "Horizon_m"], suffixes=("_fc6", "_new"))
assert len(j) == 28, f"expected 28 rows, got {len(j)}"
bad = int((j.RMSE_m_fc6 != j.RMSE_m_new).sum())
print(f"\n  regression test vs FC6 GNN rows: {28 - bad}/28 identical")
if not SMOKE:
    assert bad == 0, ("the baseline graph no longer reproduces FC6 - "
                      "something other than the adjacency has changed")

SENS.to_csv(os.path.join(OUT_DIR, f"FC9_gnn_adjacency{SUF}.csv"), index=False)
SUM = pd.DataFrame(summary)


# null control

print(f"\n  null control: {N_NULL} density-matched random directed graphs")
rng = np.random.default_rng(NULL_SEED)
k_edges = int((CCM > 0).sum())
w_lo, w_hi = CCM[CCM > 0].min(), CCM[CCM > 0].max()
off = [(i, j) for i in range(N) for j in range(N) if i != j]

null_rows = []
for t in range(N_NULL):
    A = np.zeros((N, N))
    for q in rng.choice(len(off), size=k_edges, replace=False):
        i, j = off[q]
        A[i, j] = rng.uniform(w_lo, w_hi)
    met = add_skill(evaluate(row_norm(A)))
    mean = met.groupby("Horizon_m")["skill_vs_SARIMA_%"].mean()
    null_rows.append({"trial": t, **{f"h{h}": round(mean[h], 3)
                                     for h in HORIZONS}})
    if (t + 1) % 10 == 0:
        print(f"    {t+1}/{N_NULL}")

NULL = pd.DataFrame(null_rows)
NULL.to_csv(os.path.join(OUT_DIR, f"FC9_gnn_null{SUF}.csv"), index=False)

for h in HORIZONS:
    SUM[f"null_pctile_h{h}"] = SUM[f"h{h}"].apply(
        lambda v: round(100 * (NULL[f"h{h}"] < v).mean(), 1))
SUM.to_csv(os.path.join(OUT_DIR, f"FC9_gnn_adjacency_summary{SUF}.csv"), index=False)

print("\n" + "=" * 74)
print("MEAN skill vs SARIMA (%) and percentile in the random-graph null")
print("=" * 74)
print(SUM.to_string(index=False))
print(f"\nRandom-graph null ({N_NULL} trials):")
for h in HORIZONS:
    c = NULL[f"h{h}"]
    print(f"  h={h:<2d} mean {c.mean():7.2f}  "
          f"[min {c.min():7.2f}, max {c.max():7.2f}]")
print("\nSpread across the seven real graphs (max - min):")
print("  " + "  ".join(f"h={h}: {SUM[f'h{h}'].max()-SUM[f'h{h}'].min():.1f}"
                       for h in HORIZONS))


# figure

fig, axes = plt.subplots(1, 2, figsize=(17, 6))
ax = axes[0]
for _, r in SUM.iterrows():
    ax.plot(HORIZONS, [r[f"h{h}"] for h in HORIZONS], marker="o",
            lw=2.5 if r.Graph in ("corr_abs_asbuilt", "identity") else 1.2,
            label=r.Graph)
lo = [NULL[f"h{h}"].min() for h in HORIZONS]
hi = [NULL[f"h{h}"].max() for h in HORIZONS]
ax.fill_between(HORIZONS, lo, hi, color="grey", alpha=.25,
                label=f"random graphs (min-max, n={N_NULL})")
ax.axhline(0, color="k", lw=1)
ax.set_xticks(HORIZONS)
ax.set_xlabel("horizon (months)")
ax.set_ylabel("mean skill vs SARIMA (%)")
ax.set_title("Does the GNN care which network you give it?", fontweight="bold")
ax.legend(fontsize=8)
ax.grid(alpha=.3)

ax = axes[1]
d = SENS[SENS.Horizon_m == 1].pivot(index="Lake", columns="Graph",
                                    values="skill_vs_SARIMA_%")
d[list(GRAPHS)].plot(kind="bar", ax=ax)
ax.axhline(0, color="k", lw=1)
ax.set_ylabel("skill vs SARIMA (%) at h=1")
ax.set_title("Per lake, h=1", fontweight="bold")
ax.tick_params(axis="x", rotation=45)
ax.legend(fontsize=7)
plt.suptitle("GNN adjacency sensitivity", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, f"FC9_gnn_adjacency{SUF}.png"), dpi=200)
plt.close()

print("\nWritten to", OUT_DIR)
for f in (f"FC9_gnn_adjacency{SUF}.csv", f"FC9_gnn_adjacency_summary{SUF}.csv",
          f"FC9_gnn_null{SUF}.csv", f"FC9_gnn_adjacency{SUF}.png"):
    print("   ", f)

  residual correlations: all 21 off-diagonal entries are positive (0.118 to 0.652), so |corr| discards no sign information
GNN ADJACENCY SENSITIVITY

  graphs: corr_abs_asbuilt, corr_abs_sym, corr_abs_row, ccm_directed, ccm_symmetric, corr_signed, identity, complete
  null trials: 30
    corr_abs_asbuilt h1= -15.1  h3= -27.1  h6= -14.9  h12= -13.7
    corr_abs_sym    h1= -26.1  h3= -36.5  h6= -21.1  h12=   3.5
    corr_abs_row    h1= -25.9  h3= -36.8  h6= -21.9  h12=  -1.9
    ccm_directed    h1= -30.7  h3= -44.4  h6= -32.7  h12=  -0.5
    ccm_symmetric   h1= -11.2  h3= -14.9  h6= -11.7  h12=  -2.6
    corr_signed     h1= -26.1  h3= -36.5  h6= -21.1  h12=   3.5
    identity        h1=  -3.5  h3=  -4.5  h6=  -1.8  h12=  -1.3
    complete        h1= -33.6  h3= -41.1  h6= -28.3  h12=  -2.5

  regression test vs FC6 GNN rows: 28/28 identical

  null control: 30 density-matched random directed graphs
    10/30
    20/30
    30/30

MEAN skill vs SARIMA (%) and percentile in the random-graph 